# Look-ahead · Transfer — does the K contrast survive a held-out judge?  `[EVAL]`

**Family `lookahead/transfer`.** `lookahead/reward` reports the K=0 vs K=5 contrast on the rubrics; this
family asks whether that contrast is a property of the *behaviour* or of the *grader that was optimised*.
The primary oracle (gpt-4o-mini) **was the training reward**; Claude Haiku 4.5 never played the patient and
never scored a training branch, so it is a held-out test set for every claim the primary makes. Three
questions, all read from the `data/eval_scores/` lake (free — no API calls; the paid sweep is
`notebooks/scoring/Judge_Reliability.ipynb`):

- **§1 · Cross-K pairs + sign ladder** — for each method, `LA0_In − LA5_In` at every matched iteration on
  all 9 rubrics, primary and held-out side by side (`k_pairs`), and how often the held-out judge agrees on
  the *direction*, laddered by the gap the primary reports (`k_sign_ladder`).
- **§2 · Gain retention by K** — `Δ held-out / Δ primary` of every model state over a reference base
  (`k_retention`), with K=0 and K=5 side by side at the shared endpoints (`k_retention_summary`); the figure
  `k_retention`.
- **§3 · Ledger** — `transfer_numbers.json`, every quotable cell of the four tables.

> **Judge-invariant family.** Every artifact here contains BOTH graders, so this notebook loads them itself
> via `scores_by_judge` and ignores `EDA_JUDGE`; exports carry no `<judge>/` level
> (`results/lookahead/transfer/{figures,tables}/`). Promoted 2026-08-18 from the paper generator
> `papers/2026_lookahead_pto_grpo/analysis/cross_k_multijudge.py` §1–§2 into `eda_analysis.transfer`; the
> paper's frozen `tables/cross_k_multijudge_{pairs,ladder,retention,retention_summary}.csv` are the fixture
> (means / dz / p / counts exact; the pairs' bootstrap CIs are re-drawn under `constants.BOOT_SEED`, so they
> may differ in the third decimal; retention CIs reproduce exactly). The DiD / method-gap / endpoint halves
> of that generator live in `lookahead/reward`.

**Conventions (every table repeats them in its caption).** Sign of a cross-K contrast: **+ ⇒ K=0 higher**;
`MICI` is lower-is-better, so on MICI a positive contrast favours K=5 — read the `favours_*` columns.
Pairing unit is **`persona_id`** (the trainer reshuffles the 96 personas every iteration; `file_index` is
not a pairing key). **Support is read off the data, never asserted here:** each arm's rows run to its own
last scored iteration, which can differ by grader, and every table's `iteration` column is the record of
where they stop — the setup cell derives the sentence with `constants.support_note`, which says nothing
when no arm is short. The two graders' raw scores are **never averaged** — only contrasts and ratios are combined.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 185, "display.max_columns", 50)

import os, eda_analysis
from eda_analysis import exports, plotting, transfer, reliability as R
cfg = eda_analysis.EdaConfig(family="lookahead/transfer", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # reset_results wipes figures/, incl. the banner notebook_setup just wrote — re-stamp it

# Both graders, same arm/metric filters, primary FIRST. `transfer.*` accepts these scores_long frames
# directly (persona_id already attached); the reliability (metric, model, file_index, value) shape from
# reliability.load_judge_long / load_primary_long is the other accepted input.
SC = eda_analysis.scores_by_judge(S)
PRIMARY_LABEL = eda_analysis.constants.judge_dirname("")            # 'gpt-4o-mini'
HELDOUT_TAGS = R.second_judge_tags()
if not HELDOUT_TAGS:
    raise SystemExit("no held-out judge in the score lake — run notebooks/scoring/Judge_Reliability.ipynb first")
HELDOUT_TAG = HELDOUT_TAGS[0]                                        # 'anthropic_claude-haiku-4-5'
HELDOUT_LABEL = eda_analysis.constants.judge_dirname(HELDOUT_TAG)    # 'claude-haiku-4-5'
HELDOUT_NAME = R.judge_display(HELDOUT_TAG)                          # 'Claude Haiku 4.5'
PL, JL = SC[PRIMARY_LABEL], SC[HELDOUT_LABEL]                        # primary_long / judge_long
if len(HELDOUT_TAGS) > 1:
    print(f"[transfer] NOTE: {len(HELDOUT_TAGS)} held-out judges on disk; this family reports the first ({HELDOUT_TAG}).")
PAIRING = transfer.PAIRING
# The support sentence every caption carries is DERIVED from the two frames just loaded, one per
# grader — never the static `transfer.CENSOR`, which names an arm and goes stale the moment that arm
# finishes. `constants.support_note` returns "" unless an arm really does stop before the others, so
# a caption can only report a truncation that is in the data. The legend itself is always true.
_short = [n for n in (eda_analysis.support_note(PL, prefix="", subject=f"no later state scored by {PRIMARY_LABEL}"),
                      eda_analysis.support_note(JL, prefix="", subject=f"no later state scored by {HELDOUT_LABEL}")) if n]
SUPPORT = ("Support: each arm's rows run to its own last scored iteration, which can differ by grader; "
           "every table's iteration column is the record of where they stop."
           + "".join(f" {n}" for n in dict.fromkeys(_short)))
GRADERS = (f"`primary_*` = training oracle ({PRIMARY_LABEL}); `judge_*` = held-out judge ({HELDOUT_NAME}, "
           f"never played the patient, never scored a training branch).")
print(f"primary = {PRIMARY_LABEL} {PL.shape} | held-out = {HELDOUT_LABEL} ({HELDOUT_NAME}) {JL.shape}")
print(f"[support] {SUPPORT}")

## 1 · Cross-K contrasts under both graders + the sign ladder  `[EVAL]`

**Purpose.** The measurement family (`measurement/validity`) judge-tests contrasts *within* one K — its
all-pairs table enumerates the K=5 model states, or the K=0 ones, never a K=0 vs K=5 pair. This section
does exactly that pair: for each method, `LA0_In − LA5_In` at every iteration both arms reached (iteration 0
= the two arms' INDEPENDENT base draws, a free noise-floor row) on all 9 rubrics, with the primary and the
held-out delta / *dz* / persona-bootstrap CI / Wilcoxon *p* / Holm *p* side by side. `same_sign` = the two
graders agree on direction; `judge_ci_excl0` = the held-out CI itself excludes 0. Holm is across
**iterations** within (grader, method, metric).

**The ladder** (`k_sign_ladder`) turns those contrasts (2 methods × 11 matched iterations × 9 rubrics =
198 rows on the current grid; the caption derives the count, so read it there rather than from here) into
the rate the write-up quotes — but only against an
effect size: a pooled "N % agree" reads as weak until you see the disagreements sit in gaps too small to
claim. Rungs mirror `reliability.sign_preservation` (all contrasts; |Δ primary| ≥ 0.10 / 0.25 / 0.50;
judge CI excludes 0) plus Holm rungs (primary / judge / both graders `p_holm < .05`) and `iteration ≥ 1`
(base-vs-base rows dropped), for `group` = all contrasts, each method, each metric.
⚠ Thresholds are ABSOLUTE — per-metric rows compare *down* a rubric, never across (PCT / MICI live on 0–1).

**Read.** `same_sign` is the load-bearing column, not the correlation: a level offset between judges is
expected and harmless (contrasts cancel it); what would hurt is a K *ordering* that depends on who grades.
The rows where the held-out judge flips sign on a Holm-significant primary contrast are the ones to look at.

In [ ]:
pairs = transfer.cross_k_pairs(JL, PL)
print(f"{len(pairs)} cross-K contrasts: "
      + ", ".join(f"{m} {n} rows" for m, n in pairs.groupby('method', sort=False).size().items()))

# Cross-check against the paper fixture / tracked k_paired_by_method: PTO Q1Q2 iteration 6, primary K0−K5.
_chk = pairs[(pairs.method == "PTO") & (pairs.metric == "Q1Q2") & (pairs.iteration == 6)]
if len(_chk):
    _c = _chk.iloc[0]
    print(f"[check] PTO Q1Q2 iter 6 primary K0−K5: delta {_c.primary_delta:+.3f} dz {_c.primary_dz:.3f} "
          f"(fixture +0.257, dz 0.417); held-out {_c.judge_delta:+.3f} dz {_c.judge_dz:.3f}")
    assert abs(_c.primary_delta - 0.257) < 0.002 and abs(_c.primary_dz - 0.417) < 0.002, "fixture anchor moved"

display(pairs[(pairs.metric == "Q1Q2")].round(3))
exports.save_table(pairs, "k_pairs", caption=(
    "**Cross-K contrasts under both graders.** For each method, `LA0_In − LA5_In` at every matched iteration "
    "(iteration 0 = the two arms' INDEPENDENT base draws, a noise-floor row) on all 9 rubrics. Sign: "
    "**+ => K=0 higher**; MICI is lower-is-better, so on MICI + favours K=5 — read the `favours_*` columns. "
    f"{PAIRING} {GRADERS} dz = mean/SD of persona deltas; CI = 2,000-draw percentile bootstrap over personas "
    "(seed = constants.BOOT_SEED); p = Wilcoxon; `*_p_holm` = Holm across ITERATIONS within (grader, method, "
    "metric). `same_sign` = the two graders agree on direction; `judge_ci_excl0` = the held-out CI excludes 0. "
    f"A method's rows run to the last iteration BOTH its K arms reached. {SUPPORT} Cross-check: PTO Q1Q2 iteration 6 primary = +0.257, dz 0.417 (fixture "
    "papers/2026_lookahead_pto_grpo/tables/cross_k_multijudge_pairs.csv; lookahead/reward k_paired_by_method)."))

ladder = transfer.sign_ladder(pairs)
display(ladder[ladder.group.isin(["all cross-K contrasts", "method=PTO", "method=GRPO"])])
exports.save_table(ladder, "k_sign_ladder", float_format="%.1f", caption=(
    "**Sign preservation of the cross-K contrast under the held-out judge**, as a ladder over the gap the "
    "primary oracle reports (mirrors `reliability.sign_preservation`; thresholds are ABSOLUTE, so per-metric "
    "rows compare down a rubric, never across — PCT/MICI live on 0-1). Rows are the `LA0_In − LA5_In` contrasts "
    f"of `k_pairs` ({len(pairs)} = "
    + " + ".join(f"{m} {pairs[pairs.method == m].iteration.nunique()} iteration pairs × {pairs[pairs.method == m].metric.nunique()} rubrics"
                 for m in pairs.method.unique())
    + "). Extra rungs restrict to contrasts each grader calls Holm-significant, and to iteration ≥ 1 (dropping "
    f"the base-vs-base noise-floor rows). {GRADERS} {PAIRING} {SUPPORT} The `judge CI excludes 0` rung depends on "
    "bootstrap CI bounds and can move by ±1 contrast between seeds (BOOT_SEED here vs 0 in the paper fixture)."))

## 2 · Gain retention by look-ahead K  `[EVAL]`

**Purpose.** `retention = Δ held-out / Δ primary` of each model state over a reference base is a
**train/test generalisation ratio**, not a reliability statistic: ~1 means the gain is a real behaviour change
both graders see; ~0 means it existed only in the grader that was optimised. `measurement/validity` reports it
against one shared PTO base per K; this section reports it **by K**, so the reward-hacking test can be read
as a function of the look-ahead lever.

**Reference bases (`ref_kind`).** `own_base` = the arm's OWN base draw (the headline); `method_LA0_base` /
`method_LA5_base` = the method's K=0 / K=5 base draw as a SHARED reference for both K arms (for a PTO_LA0
row `own_base` and `method_LA0_base` are the same reference and duplicate each other by design);
`eda_view_PTO_LA{K}_base` = the measurement family's convention (PTO's base of the same K), given for the
GRPO arms so `multijudge_gain_retention` numbers can be matched. Iteration-0 rows under a shared reference are
two INDEPENDENT base draws (noise floor). `retention` and its persona-bootstrap CI are suppressed (blank) where
|Δ primary| < `min_primary_delta` — 0.15 on the 1–5 / 1–7 rubrics (the `reliability.gain_retention` default)
and 0.05 on the 0–1 rate metrics PCT / MICI (their deltas are ~3× smaller; a 0.15 floor blanks almost every
MICI row). Direction-agnostic on MICI (both deltas flip together).

**Read.** Uniform retention across arms is scale compression and uninteresting — the signal is retention that
*differs by K on one metric while staying flat on another*. `k_retention_summary` puts K=0 and K=5 side by
side (own-base reference) at a fixed early anchor (iteration 5, kept for fixture comparability) and at
each K=5 arm's own endpoint, which the `iteration` column names;
`cis_disjoint` = the two K arms' retention intervals do not overlap. The figure shows the own-base
trajectories for Q1, Q2 and MICI (the harm channel); a line there can also stop early because retention was
suppressed by the |Δ primary| floor, which is a floor and not an absence of scored states.

In [ ]:
RET = transfer.retention_by_k(JL, PL)
ret, ret_sum = RET["retention"], RET["retention_summary"]
print(f"retention: {len(ret)} rows ({ret.arm.nunique()} arms × {ret.ref_kind.nunique()} reference kinds × "
      f"{ret.metric.nunique()} rubrics × iterations); summary: {len(ret_sum)} rows")

# Cross-checks against the paper fixture (retention CIs are EXACT: gain_retention seeds itself).
def _ret_at(arm, it, m, kind="own_base"):
    r = ret[(ret.arm == arm) & (ret.iteration == it) & (ret.metric == m) & (ret.ref_kind == kind)]
    return r.iloc[0] if len(r) else None
_r = _ret_at("GRPO_LA5", 5, "Q1", "eda_view_PTO_LA5_base")
if _r is not None:
    print(f"[check] GRPO_LA5 Q1 I5 retention vs PTO_LA5_Base: {_r.retention:.3f} [{_r.retention_ci_lo:.3f}, "
          f"{_r.retention_ci_hi:.3f}] (fixture 1.082 [0.936, 1.271])")
    assert abs(_r.retention - 1.082) < 0.002, "fixture anchor moved"
_r5, _r0 = _ret_at("PTO_LA5", 10, "Q2"), _ret_at("PTO_LA0", 10, "Q2")
if _r5 is not None and _r0 is not None:
    print(f"[check] PTO_LA5 Q2 I10 own-base retention {_r5.retention:.3f} (fixture 0.562); PTO_LA0 {_r0.retention:.3f} (fixture 0.849)")
    assert abs(_r5.retention - 0.562) < 0.002 and abs(_r0.retention - 0.849) < 0.002, "fixture anchor moved"

# Support for THIS frame, derived: the always-true legend, plus a per-arm sentence ONLY if an arm's
# retention rows really do stop before the others (`support_note` returns "" when none does).
_ret_short = eda_analysis.support_note(ret, prefix="", subject="no later scored state in this frame")
SUPPORT_RET = SUPPORT + (f" {_ret_short}" if _ret_short else "")

display(ret_sum.round(3))
exports.save_table(ret, "k_retention", caption=(
    "**Gain retention by look-ahead K.** `retention = Δ held-out / Δ primary` of each model state over a "
    "reference base — the train/test generalisation ratio (~1 = the gain is real to a judge that never played "
    "the patient; ~0 = it existed only in the optimised grader). `ref_kind`: `own_base` = the arm's OWN base "
    "draw; `method_LA0_base` / `method_LA5_base` = the method's K=0 / K=5 base draw as a SHARED reference for "
    "both K arms (for a PTO_LA0 row, `own_base` and `method_LA0_base` are the same reference and duplicate "
    "each other by design); `eda_view_PTO_LA{K}_base` = the measurement family's convention (PTO's base of the "
    "same K), given for the GRPO arms so measurement/validity multijudge_gain_retention numbers can be matched. "
    "Iteration-0 rows under a shared reference are two INDEPENDENT base draws (noise floor). `retention` and its "
    "CI are suppressed (blank) where |Δ primary| < `min_primary_delta` — 0.15 on the 1-5 / 1-7 rubrics (the "
    "`reliability.gain_retention` default, whose persona-bootstrap CI this is) and 0.05 on the 0-1 rate metrics "
    f"PCT/MICI (their deltas are ~3x smaller; a 0.15 floor blanks almost every MICI row). Δ held-out = {HELDOUT_NAME} "
    f"gain over base; Δ primary = {PRIMARY_LABEL} (training oracle) gain over base. {PAIRING} Direction-agnostic on "
    f"MICI (both deltas flip together). {SUPPORT_RET} Cross-check: GRPO_LA5 Q1 iteration 5 vs eda_view_PTO_LA5_base = "
    "1.082 [0.936, 1.271]; PTO_LA5 Q2 iteration 10 own-base = 0.562 vs PTO_LA0 0.849 (fixture "
    "papers/2026_lookahead_pto_grpo/tables/cross_k_multijudge_retention.csv). The .md is a head excerpt when the "
    "table exceeds the markdown ceiling — every row is on the `transfer.xlsx` sheet and in `transfer_numbers.json`."))
exports.save_table(ret_sum, "k_retention_summary", caption=(
    "**Gain retention, K=0 vs K=5 side by side (own-base reference)** at a fixed early anchor (iteration 5, "
    "kept for comparability with the frozen fixture) and at each K=5 arm's own endpoint — the `iteration` "
    "column names that endpoint for each method. retention = Δ held-out "
    f"({HELDOUT_NAME}) / Δ primary ({PRIMARY_LABEL}, the training oracle) over the arm's own base; CI = persona "
    "bootstrap; `cis_disjoint` = the two K arms' retention intervals do not overlap. Blank where |Δ primary| is "
    f"under the floor (0.15 on Q1Q2/Q1/Q2/MITI, 0.05 on MICI). {PAIRING} {SUPPORT_RET} Full table: `k_retention`."))

fig = plotting.k_retention(ret, metrics=("Q1", "Q2", "MICI"), palette=S.PALETTE,
                           heldout_label=HELDOUT_NAME, primary_label=f"{PRIMARY_LABEL} (training oracle)")
if fig is not None:
    exports.save_fig(fig, "k_retention", caption=(
        "**Gain retention by K, own-base reference.** Rows = Q1, Q2, MICI (harm channel); columns = PTO, GRPO. "
        f"retention = Δ held-out ({HELDOUT_NAME}) / Δ primary ({PRIMARY_LABEL}, the training oracle) over each "
        "arm's OWN base; K=0 solid + circle, K=5 dashed + square, ribbons = persona-bootstrap 95% CI, dash-dotted "
        "line = retention 1 (the gain is fully real to the held-out judge). Blank where |Δ primary| < 0.15 (Q1/Q2) "
        "or < 0.05 (MICI) — a line that stops or gaps is that floor, not a missing model state. "
        f"{PAIRING} {SUPPORT_RET} Table: `k_retention`."))
    plt.show()

## 3 · Ledger  `[EVAL]`

Every quotable cell of the four tables, as `results/lookahead/transfer/tables/transfer_numbers.json`
(`{dotted.key: {value, source, note}}`, the same key families as the paper's frozen
`analysis/out/cross_k_multijudge.json`: `kcontrast.<method>.<metric>.iter<n>`, `ladder.<group>.<subset>`,
`retention.<arm>.<metric>.iter<n>.<ref_kind>`, `retention_summary.<method>.<metric>.iter<n>`). A paper's
`NUMBERS.md` cites `transfer_numbers.json :: <key>` rather than re-typing a number from a table.

In [ ]:
NUM = transfer.transfer_numbers(pairs=pairs, ladder=ladder, retention=ret, retention_summary=ret_sum)
path = exports.save_numbers("transfer_numbers", NUM, caption=(
    "**Number ledger for the transfer family** — every cell of `k_pairs` (kcontrast.*), `k_sign_ladder` "
    "(ladder.*), `k_retention` (retention.*) and `k_retention_summary` (retention_summary.*), as "
    "`{value, source, note}` records keyed like the paper's cross_k_multijudge.json. Sign: + => K=0 higher; MICI "
    f"lower-better. {PAIRING} {SUPPORT_RET}"))
print(f"{len(NUM)} keys -> {os.path.relpath(path, S.RESULTS_DIR)}")

## 4 · How to read this family
- **`same_sign` (§1) is the load-bearing number, not the correlation.** A level offset between graders is
  expected and harmless — the thesis reports contrasts, which cancel it. What would hurt is a K ordering that
  depends on who grades. Read the ladder rung by rung: agreement on the contrasts each grader calls
  Holm-significant is the claim; agreement on the base-vs-base rows is noise.
- **Sign is + ⇒ K=0 higher.** On MICI (lower = better) a positive contrast favours K=5; every table carries
  `favours_*` so the reader never has to flip it by hand.
- **Retention (§2) is a train/test ratio.** The primary WAS the reward; the held-out judge never touched
  training. ~1 = a real behaviour change both graders see; ~0 = a gain that lived only in the optimised grader.
  Uniform retention is scale compression — the signal is retention that differs by K on one rubric and not on
  another (Q2 vs Q1 at the PTO endpoint is the row to look at).
- **Blank retention cells are a floor, not an absence.** Where the primary's own gain is under
  `min_primary_delta`, a ratio would be noise over noise; the row is kept, the ratio is suppressed.
- **Never average the two graders' raw scores.** `transfer.py` combines only contrasts and ratios.
- **Support.** Read each arm's endpoint off the `iteration` column of the table in hand — a cross-K row exists
  at every iteration BOTH K arms of that method reached, and the support sentence in each caption is derived
  from the frame (`constants.support_note`), so it names an arm only when that arm genuinely stops short.
- **Iteration ≠ spend.** A K=5 iteration costs more than a K=0 one, so an iteration-matched row is not a
  budget-matched one: GRPO K=5 spent 51.205 GPU-h against K=0's 27.906 (51.205 / 27.906 = 1.84×), and PTO K=5
  19.681 against 8.119 (`results/compute/cost/tables/compute_by_arm.md`). The budget-matched reading of the
  same lever is `compute/cost`, and its `budget_sweep` — not a single iso-compute row — is what to quote.
- _(The measured values are narrated in `results/lookahead/SUMMARY.md`; the caveats they imply in
  `results/LIMITATIONS.md`. This notebook is where they are computed.)_

In [ ]:
exports.prune_orphan_captions(); print("index ->", exports.build_index())